# Student Performance Analytics & ML Modeling

This notebook covers:
1. **Exploratory Data Analysis (EDA)**: Demographic breakdown, study habits, and subject performance.
2. **Correlation & Factor Analysis**: Impact of attendance, study hours, and test prep on final scores.
3. **Feature Engineering**: Creating engagement and STEM indices.
4. **Model Training & Comparison**: Ridge Regression, Random Forest, and Gradient Boosting.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load raw dataset
data_path = os.path.join('..', 'data', 'raw', 'student_performance_data.csv')
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
df.head()

## 1. Summary Statistics & Data Info

In [ ]:
df.info()
df.describe().round(2)

## 2. Score Distributions & Risk Categories

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['final_score'], kde=True, ax=axes[0], color='#4f46e5', bins=25)
axes[0].set_title('Distribution of Final Exam Scores')
axes[0].set_xlabel('Final Score (0-100)')

risk_counts = df['risk_level'].value_counts()
axes[1].pie(risk_counts, labels=risk_counts.index, autopct='%1.1f%%', colors=['#10b981', '#f59e0b', '#ef4444'])
axes[1].set_title('Student Risk Profile Breakdown')

plt.tight_layout()
plt.show()

## 3. Impact of Attendance and Study Hours on Scores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=df, x='attendance_rate', y='final_score', hue='risk_level', palette='Set2', ax=axes[0])
axes[0].set_title('Attendance Rate vs Final Score')
axes[0].axvline(75, color='red', linestyle='--', alpha=0.7, label='75% Minimum Attendance')
axes[0].legend()

sns.scatterplot(data=df, x='study_hours_per_week', y='final_score', hue='test_prep_course', palette='coolwarm', ax=axes[1])
axes[1].set_title('Study Hours/Week vs Final Score')

plt.tight_layout()
plt.show()

## 4. Correlation Matrix

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(10, 8))
sns.heatmap(numeric_df.corr(), annot=True, cmap='Blues', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.show()

## 5. Machine Learning Modeling Baseline

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# Feature subset
X = df[['study_hours_per_week', 'attendance_rate', 'previous_score', 'math_score', 'science_score', 'english_score']]
y = df['final_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print(f"Baseline Random Forest R2: {r2_score(y_test, y_pred):.4f}")
print(f"Baseline Random Forest RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")